# 07. 증상 3개만 수정한 데이터로 다시 학습 (A100, 약 15~20분)

06 결과에서 수정 효과가 있었던 **구토·오심, 두통, 어지러움**만 Qwen이 고친 라벨을 쓰고,
나머지 5개(고열, 복통, 열상, 전신쇠약, 호흡곤란)는 **원래 라벨**로 되돌려 모델 하나를 다시 학습한다.

- Qwen 판정은 다시 하지 않는다. 06에서 저장한 `judge/val_judged.csv`, `judge/train_judged.csv`를 읽는다.
- 원본 모델도 다시 학습하지 않는다. 06의 `원본모델_val_prob.npy`를 불러와 비교한다.
- 새 모델 이름은 `정리모델_3개` (06의 `정리모델.pt`와 겹치지 않게).
- 결과: **`results/07_결과/`** 한 폴더 + `models/정리모델_3개.pt`.
- 06 결과, 원본 JSON, `labels_df.pkl`은 읽기만 한다.

**돌리는 사람에게**
1. 런타임 → 런타임 유형 변경 → **A100** (T4도 되지만 학습이 1시간 넘게 걸릴 수 있다).
2. 런타임 → 모두 실행.
3. 끊기면 그냥 다시 실행: 학습이 끝난 모델은 건너뛴다.

## 0. 준비

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
if not os.path.isdir(f"{PROJECT_ROOT}/src"):   # 못 찾으면 드라이브에서 자동으로 찾기
    found = [p for pat in ["/content/drive/**/증상인식_최종_프로젝트*", "/content/drive/.shortcut-targets-by-id/*/증상인식_최종_프로젝트*"]
             for p in glob.glob(pat, recursive=True) if os.path.isdir(f"{p}/src")]
    assert found, "프로젝트 폴더를 못 찾음. 공유 폴더를 '내 드라이브'에 바로가기로 추가했는지 확인"
    PROJECT_ROOT = found[0]
print("프로젝트 폴더:", PROJECT_ROOT)
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from src import config
from src.config import TrainConfig
from src.data import load_labels, prep, symptom_matrix
from src.cleaning import decide_actions, action_summary, build_fixed, save_fixed
from src.model import encode_chunks
from src.trainer import seed_all, train_model, predict
from src.infer import save_model
from src.evaluate import claude_fixed_val, score_table, log_experiment
from src.metrics import pr_report

jcfg = config.JudgeConfig()
cfg = TrainConfig()
OUT = f"{config.RESULTS_DIR}/07_결과"     # 이번 결과를 모으는 폴더
OUT06 = f"{config.RESULTS_DIR}/06_결과"   # 06 결과(원본 모델 확률을 가져옴)
NAME = "정리모델_3개"
os.makedirs(OUT, exist_ok=True)
# 06 판정 표: judge/ 폴더에 없으면 06_결과 폴더에 복사해 둔 것을 쓴다
JUDGED = {k: next((p for p in [f"{config.JUDGE_DIR}/{k}_judged.csv", f"{OUT06}/{k}_judged.csv"] if os.path.exists(p)), None)
          for k in ["val", "train"]}
assert all(JUDGED.values()), f"06 판정 표가 없음: {JUDGED}"
assert os.path.exists(f"{OUT06}/원본모델_val_prob.npy"), "06 원본 모델 확률이 없음"
print("판정 표:", JUDGED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(GPU 런타임으로 바꾸기)")

## 1. 증상 3개만 수정한 라벨 만들기
`MODES`: "수정" = Qwen 규칙대로 고치고 애매한 칸은 삭제 / "안 함" = 원래 라벨 그대로.

In [ ]:
FIX = ["구토·오심", "두통", "어지러움"]
MODES = {s: ("수정" if s in FIX else "안 함") for s in config.LABELS}
print(MODES)

read = lambda p: pd.read_csv(p, dtype={"대화ID": str})
va_j = decide_actions(read(JUDGED["val"]), jcfg, MODES)
tr_j = decide_actions(read(JUDGED["train"]), jcfg, MODES)

tr_df, va_df = load_labels()
s_tr = action_summary(tr_j, tr_df, jcfg.warn_min_pos_train, jcfg)
s_va = action_summary(va_j, va_df, jcfg.warn_min_pos_val, jcfg)
pd.concat({"Train": s_tr, "Val": s_va}, axis=1).to_csv(f"{OUT}/처리_분포표.csv", encoding="utf-8-sig")
print("[Train]"); display(s_tr)
print("[Val]"); display(s_va)

In [ ]:
# 고친 라벨 + 삭제 마스크 저장 (06과 같은 행 순서). 06의 train_fixed.npz는 건드리지 않는다
tok = AutoTokenizer.from_pretrained(cfg.plm_name)
tr = prep(tr_df, tok.sep_token, drop_empty=True)
va = prep(va_df, tok.sep_token, drop_empty=False)
print("Train"); Y_tr, M_tr = build_fixed(tr, tr_j)
print("Val");   Y_va, M_va = build_fixed(va, va_j)
save_fixed(f"{config.DATA_DIR}/train_fixed_3개.npz", tr, Y_tr, M_tr)
save_fixed(f"{config.DATA_DIR}/val_fixed_3개.npz", va, Y_va, M_va)
Y_tr0, Y_va0 = symptom_matrix(tr["symptom"]), symptom_matrix(va["symptom"])   # 원래 라벨
print(pd.DataFrame({"Train 원래 있음": Y_tr0.sum(0), "Train 최종 있음": (Y_tr * M_tr).sum(0),
                    "Val 원래 있음": Y_va0.sum(0), "Val 최종 있음": (Y_va * M_va).sum(0)},
                   index=config.LABELS).astype(int).to_string())

## 2. KoELECTRA 학습 → 모델 저장 → Val 확률 저장
학습 설정은 06과 같다(seed 42, 3 epoch). 끝난 모델은 다시 실행해도 건너뛴다.

In [ ]:
prob_path = f"{OUT}/{NAME}_val_prob.npy"
if os.path.exists(prob_path):
    print(NAME, "이미 끝남 → 건너뜀")
    p_new = np.load(prob_path)
else:
    ch_tr = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(tr["text"], desc="Train")]
    ch_va = [encode_chunks(t, tok, cfg.max_len, cfg.stride) for t in tqdm(va["text"], desc="Val")]
    seed_all(cfg.seed)
    model = train_model(ch_tr, Y_tr, cfg, tok, M_tr, desc=NAME, loss_csv=f"{OUT}/{NAME}_loss.csv")
    save_model(model, f"{config.MODEL_DIR}/{NAME}.pt", config.LABELS, cfg)
    p_new = predict(model, ch_va, tok, desc=f"{NAME} Val")
    np.save(prob_path, p_new)
    del model; torch.cuda.empty_cache()

probs = {"3개 수정 모델": p_new, "원본 모델": np.load(f"{OUT06}/원본모델_val_prob.npy")}
if os.path.exists(f"{OUT06}/정리모델_val_prob.npy"):
    probs["전부 수정 모델(06)"] = np.load(f"{OUT06}/정리모델_val_prob.npy")
assert all(len(p) == len(va) for p in probs.values()), "06 확률과 Val 행 수가 다름"

## 3. 점수표 (Macro F1, 기준값 0.5) + 실험기록
가장 중요한 비교는 **고친 Val**(Claude가 고친 Val, Qwen과 따로 만든 기준)이다.
'정리 Val'은 이번 3개 수정 규칙으로 고친 Val이라 06의 '정리 Val'과 다르다.

In [ ]:
val_sets = {"정리 Val": (Y_va, M_va), "원본 Val": (Y_va0, None)}
fixed = claude_fixed_val(va_df)
if fixed is not None:
    val_sets["고친 Val"] = fixed
macro, detail = score_table(probs, val_sets)
macro.to_csv(f"{OUT}/점수_macro.csv", encoding="utf-8-sig")
detail.to_csv(f"{OUT}/점수_증상별.csv", encoding="utf-8-sig")
display(macro)
display(detail)

main_val = "고친 Val" if "고친 Val" in val_sets else "정리 Val"
Y_m, M_m = val_sets[main_val]
pr = pd.concat({m: pr_report(Y_m, probs[m], config.LABELS, 0.5, M_m) for m in probs}, axis=1)
pr.to_csv(f"{OUT}/재현율_정밀도_{main_val.replace(' ', '')}.csv", encoding="utf-8-sig")
print(f"[{main_val}] 재현율·정밀도"); display(pr)

In [ ]:
det = {}
for v, (Y, M) in val_sets.items():
    t = pr_report(Y, p_new, config.LABELS, 0.5, M)
    t["F1"] = detail[f"3개 수정 모델 / {v}"].reindex(t.index)
    det[v] = t
log = log_experiment(f"{config.RESULTS_DIR}/04_final/실험기록.csv", "3개 수정 모델",
                     "구토·오심·두통·어지러움만 Qwen 수정, 나머지 원래 라벨, 기본 설정",
                     macro.loc["3개 수정 모델"].to_dict(), det)
display(log[[c for c in log.columns if "Macro" in c or c in ("실험", "바꾼 설정")]])

## 4. 결과 모으기

In [ ]:
import shutil
for p in [f"{config.DATA_DIR}/train_fixed_3개.npz", f"{config.DATA_DIR}/val_fixed_3개.npz",
          f"{config.RESULTS_DIR}/04_final/실험기록.csv"]:
    if os.path.exists(p):
        shutil.copy(p, OUT)
print("결과 폴더:", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(f'{OUT}/{f}') / 1e6:.1f}MB)")
print("모델:", f"{config.MODEL_DIR}/{NAME}.pt")